# Geotagger Evaluation

Calls `/geotag` for each fixture and measures:
- **City precision** — when expected_city is set, did we return the correct city?
- **Null precision** — when no city is expected, did we correctly return null?
- **Place recall** — did we find at least the expected minimum number of places?

**Prerequisite:** NLP service running at `http://localhost:8001` (`/readyz` returns 200).

In [1]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('geotag_cases.json')
print(f'Loaded {len(cases)} test cases')

Loaded 8 test cases


In [2]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, 'Service not ready'

In [3]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/geotag',
        json={
            'article_id': case['article_id'],
            'text': case['text'],
            'headline': case.get('headline', ''),
        }, 
        headers=HEADERS
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, f"{case['id']}: HTTP {resp.status_code}"
    data = resp.json()

    expected_city = case.get('expected_city')
    expected_present = case.get('expected_city_present', True)
    expected_places_min = case.get('expected_places_min', 0)

    city_correct = (
        (expected_city is None and data['city'] is None) or
        (expected_city is not None and data['city'] == expected_city)
    )
    places_ok = len(data.get('all_places', [])) >= expected_places_min
    passed = city_correct and places_ok
    icon = '✅' if passed else '❌'

    results.append({
        'id': case['id'],
        'description': case['description'],
        'expected_city': expected_city,
        'returned_city': data['city'],
        'city_confidence': data.get('city_confidence', 0),
        'city_correct': city_correct,
        'places_found': len(data.get('all_places', [])),
        'places_ok': places_ok,
        'latency_s': latency,
        'passed': passed,
    })

    print(f"{icon} [{case['id']}] city={data['city']!r} (expected={expected_city!r})  conf={data.get('city_confidence',0):.2f}  places={len(data.get('all_places',[]))}  {latency:.1f}s")
    if not city_correct:
        print(f"   ⚠ CITY MISMATCH — description: {case['description']}")
    print()

❌ [geo-001] city=None (expected='Madrid')  conf=0.00  places=2  0.2s
   ⚠ CITY MISMATCH — description: Single unambiguous city mention

❌ [geo-002] city=None (expected='Valladolid')  conf=0.00  places=1  0.2s
   ⚠ CITY MISMATCH — description: City with accent (common Spanish name)

❌ [geo-003] city='Madrid' (expected='Barcelona')  conf=0.58  places=4  0.2s
   ⚠ CITY MISMATCH — description: Multiple city mentions — should pick the dominant one

❌ [geo-004] city=None (expected='Barcelona')  conf=0.00  places=2  0.1s
   ⚠ CITY MISMATCH — description: Street mention without explicit city — city inferred from DB snapshot

✅ [geo-005] city=None (expected=None)  conf=0.00  places=1  0.2s

❌ [geo-006] city=None (expected='Barcelona')  conf=0.00  places=1  0.2s
   ⚠ CITY MISMATCH — description: Ambiguous place name that could be multiple cities

❌ [geo-007] city=None (expected='Sevilla')  conf=0.00  places=0  0.2s
   ⚠ CITY MISMATCH — description: City mentioned only in headline, not body

✅ [g

In [4]:
passing = [r for r in results if r['passed']]
city_precision = sum(1 for r in results if r['city_correct']) / len(results)
avg_conf = sum(r['city_confidence'] for r in results) / len(results)
avg_lat = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('GEOTAGGER', {
    'Cases': len(results),
    'Fully passing': f"{len(passing)}/{len(results)}",
    'City precision': city_precision,
    'Target precision': 0.85,
    'Average city confidence': avg_conf,
    'Average latency (s)': avg_lat,
})


  GEOTAGGER
  Cases                               8
  Fully passing                       2/8
  City precision                      0.250
  Target precision                    0.850
  Average city confidence             0.103
  Average latency (s)                 0.170



## Tuning Guide

| Symptom | Lever | Where |
|---------|-------|-------|
| Wrong city returned for ambiguous articles | Increase weight of headline city mentions | `nlp/geotagger/disambiguator.py: _match_city` |
| City not found when name uses accent | Check `_normalize()` strips accents correctly | `nlp/geotagger/gazetteer.py: _normalize` |
| Neighbourhood resolves as wrong city | Add neighbourhood→city mapping to `cities_snapshot.json` | `scripts/snapshot_cities.py` |
| Low `city_confidence` on unambiguous articles | Review `_score()` formula in disambiguator | `nlp/geotagger/disambiguator.py` |
| City returned when none expected | Raise the confidence floor before setting `city` field | `nlp/geotagger/service.py` |
| GeoNames entries missing | Re-run `build_geonames_es.py` with a lower population filter | `scripts/build_geonames_es.py: --min-population` |